# 🗣️ Orpheus-3B — Batch Voice Cloning Test

Runs **12 test texts** through Orpheus in a single execution.
Saves all outputs to `outputs/orpheus/` and zips for download.

## ▶️ Instructions
1. **Runtime → Change runtime type → GPU** (T4 is enough with 4-bit)
2. Run all cells top to bottom
3. Upload your reference voice when prompted
4. Wait for all 12 texts to generate
5. Download the zip at the end

## 1. Setup and Installation

In [ ]:
import sys, subprocess, shutil

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("unsloth")
pip_install("snac", "soundfile", "librosa")

if shutil.which("ffmpeg") is None:
    print("⚠️ ffmpeg not found. WAV will still work, but MP3/M4A may not.")
else:
    print("✅ ffmpeg found:", shutil.which("ffmpeg"))
print("✅ Install step finished.")

## 2. Imports

In [ ]:
import os, gc, warnings, time, json
import numpy as np
import torch

warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. This notebook needs a GPU.\n"
        "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("   torch:", torch.__version__)

## 3. Configuration + Test Texts

In [ ]:
# --- Model -------------------------------------------------------------------
MODEL_NAME      = "unsloth/orpheus-3b-0.1-pretrained"
LOAD_IN_4BIT    = True
MAX_SEQ_LENGTH  = 8192

# --- Reference voice ---------------------------------------------------------
REF_AUDIO_PATH  = ""      # leave "" to upload in Colab
REF_TRANSCRIPT  = "Replace this text with the exact words spoken in your reference audio sample."

# --- Audio preprocessing -----------------------------------------------------
TARGET_SAMPLE_RATE = 24000
MIN_DURATION_SEC   = 20
MAX_DURATION_SEC   = 30
HARD_MIN_SEC       = 15
TRIM_TOP_DB        = 30
PEAK_NORM          = 0.95

# --- Generation --------------------------------------------------------------
TEMPERATURE        = 0.6
TOP_P              = 0.95
REPETITION_PENALTY = 1.1
MAX_NEW_TOKENS     = 1200
PRIME_SPEECH       = True

# --- 12 Test Texts -----------------------------------------------------------
TEST_TEXTS = [
    {"id": 1, "text": "Your ticket number is B 4 7 2 9 and the fare is rupees three thousand two hundred.", "language": "English", "category": "Booking Confirmation"},
    {"id": 2, "text": "Thank you for calling customer support. Your query has been registered and our team will get back to you within twenty four hours. We apologise for the inconvenience caused.", "language": "English", "category": "Customer Support"},
    {"id": 3, "text": "Departure at 06:45 AM on 3rd February 2025", "language": "English", "category": "Flight Details"},
    {"id": 4, "text": "Aapki booking confirm ho gayi. Reference number note kar lijiye: B 4 9 2 1.", "language": "Hindi", "category": "Booking Confirmation"},
    {"id": 5, "text": "Namaskar aur hamare service mein aapka swagat hai. Aapka loan application approved ho gaya hai. Amount aapke registered account mein do se teen working days mein credit ho jayega. Kisi bhi sahayta ke liye humse contact karein.", "language": "Hindi", "category": "Loan / Finance"},
    {"id": 6, "text": "Flight booking ke liye 1 dabayen. Flight status ke liye 2 dabayen. Cancellation ke liye 3 dabayen.", "language": "Hindi", "category": "IVR Menu"},
    {"id": 7, "text": "Dhanyavaad IndiGo ko call karne ke liye. Aapka din mangalmay ho.", "language": "Hindi", "category": "Call Closing"},
    {"id": 8, "text": "Aapka PNR number hai A B 1 2 3 4. Ise save kar lijiye.", "language": "Hindi", "category": "Booking Confirmation"},
    {"id": 9, "text": "Kya aap travel insurance add karna chahenge? Yeh sirf rupees 299 mein available hai.", "language": "Hindi", "category": "Upsell / Add-on"},
    {"id": 10, "text": "Yeh final boarding call hai passengers Mr. Sharma aur Mrs. Gupta ke liye, flight 6E 888 ke liye gate C 3 par.", "language": "Hindi", "category": "Boarding Announcement"},
    {"id": 11, "text": "IndiGo BluChip Gold members aur business class passengers priority boarding le sakte hain.", "language": "Hindi", "category": "Boarding Announcement"},
    {"id": 12, "text": "IndiGo wallet mein minimum rupees 500 add kar sakte hain future bookings ke liye.", "language": "Hindi", "category": "Wallet / Payment"}
]

OUTPUT_DIR = "outputs/orpheus"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config loaded. {len(TEST_TEXTS)} test texts ready.")
for t in TEST_TEXTS:
    print(f"  [{t['id']:2d}] [{t['language']:7s}] [{t['category']:22s}] {t['text'][:60]}...")

## 4. Upload / Load Voice Sample

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not REF_AUDIO_PATH:
    if IN_COLAB:
        print("Upload your 20-30s voice sample (WAV/MP3/M4A/FLAC):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        REF_AUDIO_PATH = list(uploaded.keys())[0]
    else:
        raise RuntimeError(
            "REF_AUDIO_PATH is empty and not running in Colab.\n"
            "Set REF_AUDIO_PATH in the Configuration cell to your audio file."
        )

if not os.path.exists(REF_AUDIO_PATH):
    raise FileNotFoundError(f"Audio file not found: {REF_AUDIO_PATH}")
print("✅ Reference audio:", REF_AUDIO_PATH)

## 5. Audio Preprocessing

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

def load_and_preprocess(path):
    """Return a float32 mono waveform at TARGET_SAMPLE_RATE, cleaned up."""
    try:
        wav, sr = librosa.load(path, sr=TARGET_SAMPLE_RATE, mono=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not decode '{path}'. Unsupported format or missing ffmpeg.\n"
            f"Original error: {e}"
        )
    wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 0:
        wav = (wav / peak) * PEAK_NORM
    wav = wav.astype(np.float32)
    return wav, TARGET_SAMPLE_RATE

ref_wav, ref_sr = load_and_preprocess(REF_AUDIO_PATH)
duration = len(ref_wav) / ref_sr
print(f"Processed: {duration:.1f}s, mono, {ref_sr} Hz, {len(ref_wav)} samples")

if duration < HARD_MIN_SEC:
    print(f"🚨 WARNING: only {duration:.1f}s — too short.")
elif duration < MIN_DURATION_SEC:
    print(f"⚠️ {duration:.1f}s is a little short; {MIN_DURATION_SEC}-{MAX_DURATION_SEC}s is ideal.")
elif duration > MAX_DURATION_SEC:
    print(f"⚠️ {duration:.1f}s is longer than {MAX_DURATION_SEC}s; consider trimming.")
else:
    print("✅ Duration is in the ideal 20-30s window.")

print("Preview of your processed reference:")
display(Audio(ref_wav, rate=ref_sr))

## 6. Model Loading

In [ ]:
from unsloth import FastLanguageModel
try:
    from unsloth import FastModel
except Exception:
    FastModel = None

model = tokenizer = None
load_err = None
for loader_name, loader in [("FastLanguageModel", FastLanguageModel),
                            ("FastModel", FastModel)]:
    if loader is None:
        continue
    try:
        model, tokenizer = loader.from_pretrained(
            model_name      = MODEL_NAME,
            max_seq_length  = MAX_SEQ_LENGTH,
            dtype           = None,
            load_in_4bit    = LOAD_IN_4BIT,
        )
        print(f"✅ Loaded via {loader_name}.")
        break
    except torch.cuda.OutOfMemoryError:
        raise RuntimeError("CUDA OOM while loading the model.")
    except TypeError as e:
        load_err = e
        continue
    except Exception as e:
        raise RuntimeError(f"Model download/load failed: {e}")

if model is None:
    raise RuntimeError(f"Could not load {MODEL_NAME}. Last error: {load_err}")

try:
    FastLanguageModel.for_inference(model)
except Exception:
    pass
model.eval()
print("✅ Orpheus ready.")

from snac import SNAC
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda").eval()
print("✅ SNAC 24kHz loaded.")

## 7. Encode Reference Audio

In [ ]:
# --- Orpheus special-token constants ------------------------------------------
tokeniser_length = 128256
start_of_text  = 128000
end_of_text    = 128009
start_of_speech = tokeniser_length + 1
end_of_speech   = tokeniser_length + 2
start_of_human  = tokeniser_length + 3
end_of_human    = tokeniser_length + 4
start_of_ai     = tokeniser_length + 5
end_of_ai       = tokeniser_length + 6
pad_token       = tokeniser_length + 7
audio_tokens_start = tokeniser_length + 10

def tokenise_audio(waveform_np, orig_sr=TARGET_SAMPLE_RATE):
    """Encode a mono float32 waveform into Orpheus audio tokens (7 per frame)."""
    if orig_sr != 24000:
        waveform_np = librosa.resample(
            np.asarray(waveform_np, dtype=np.float32), orig_sr=orig_sr, target_sr=24000)
    wav = torch.from_numpy(np.asarray(waveform_np, dtype=np.float32)).unsqueeze(0)
    wav = wav.unsqueeze(0).to("cuda")
    with torch.inference_mode():
        codes = snac_model.encode(wav)
    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item()       + 128266)
        all_codes.append(codes[1][0][2*i].item()     + 128266 + 4096)
        all_codes.append(codes[2][0][4*i].item()     + 128266 + (2*4096))
        all_codes.append(codes[2][0][(4*i)+1].item() + 128266 + (3*4096))
        all_codes.append(codes[1][0][(2*i)+1].item() + 128266 + (4*4096))
        all_codes.append(codes[2][0][(4*i)+2].item() + 128266 + (5*4096))
        all_codes.append(codes[2][0][(4*i)+3].item() + 128266 + (6*4096))
    return all_codes

def redistribute_codes(code_list):
    """Inverse of tokenise_audio: split flat tokens into SNAC's 3 layers, decode."""
    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(code_list) + 1) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - (2*4096))
        layer_3.append(code_list[7*i + 3] - (3*4096))
        layer_2.append(code_list[7*i + 4] - (4*4096))
        layer_3.append(code_list[7*i + 5] - (5*4096))
        layer_3.append(code_list[7*i + 6] - (6*4096))
    codes = [torch.tensor(layer_1).unsqueeze(0),
             torch.tensor(layer_2).unsqueeze(0),
             torch.tensor(layer_3).unsqueeze(0)]
    with torch.inference_mode():
        audio_hat = snac_model.decode(codes)
    return audio_hat

ref_codes = tokenise_audio(ref_wav, orig_sr=TARGET_SAMPLE_RATE)
print(f"Reference encoded: {len(ref_codes)} tokens ({len(ref_codes)//7} frames, "
      f"~{len(ref_codes)/150:.1f}s of codec audio)")

## 8. 🔄 Batch Generation — All 12 Test Texts

This cell loops through all 12 test texts, generates audio for each,
and saves the outputs. **No need to re-run anything manually!**

In [ ]:
results = []

ref_text_ids = tokenizer.encode(REF_TRANSCRIPT, add_special_tokens=True)
ref_text_ids.append(end_of_text)

print(f"\n{'='*70}")
print(f"  BATCH GENERATION: {len(TEST_TEXTS)} texts through Orpheus")
print(f"{'='*70}\n")

for idx, test in enumerate(TEST_TEXTS):
    test_id = test["id"]
    text = test["text"]
    wav_filename = f"test_{test_id:02d}.wav"
    wav_path = os.path.join(OUTPUT_DIR, wav_filename)

    print(f"\n--- [{idx+1}/{len(TEST_TEXTS)}] Test #{test_id}: {test['category']} ({test['language']}) ---")
    print(f"    Text: {text[:80]}{'...' if len(text)>80 else ''}")

    start_time = time.time()
    status = "success"
    error_msg = None

    try:
        # Build prompt
        tgt_text_ids = tokenizer.encode(text, add_special_tokens=True)
        tgt_text_ids.append(end_of_text)

        prompt_ids = (
            [start_of_human] + ref_text_ids + [end_of_human]
            + [start_of_ai] + [start_of_speech] + ref_codes + [end_of_speech] + [end_of_ai]
            + [start_of_human] + tgt_text_ids + [end_of_human]
        )
        if PRIME_SPEECH:
            prompt_ids += [start_of_ai, start_of_speech]

        input_ids = torch.tensor([prompt_ids], dtype=torch.int64).to("cuda")
        attention_mask = torch.ones_like(input_ids)

        if input_ids.shape[1] + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
            print(f"    ⚠️ prompt + generation may exceed max_seq_length")

        # Generate
        with torch.inference_mode():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                repetition_penalty=REPETITION_PENALTY,
                num_return_sequences=1,
                eos_token_id=end_of_speech,
                use_cache=True,
            )

        # Parse tokens
        token_to_find, token_to_remove = 128257, 128258
        idx_found = (generated_ids == token_to_find).nonzero(as_tuple=True)
        if len(idx_found[1]) > 0:
            last = idx_found[1][-1].item()
            cropped = generated_ids[:, last + 1:]
        else:
            cropped = generated_ids

        row = cropped[0]
        row = row[row != token_to_remove]
        new_len = (row.size(0) // 7) * 7
        row = row[:new_len]
        code_list = [t.item() - 128266 for t in row]

        if len(code_list) == 0:
            raise RuntimeError("No audio tokens generated.")

        # Decode — move SNAC to CPU for decoding
        snac_model.to("cpu")
        audio_hat = redistribute_codes(code_list)
        audio_np = audio_hat.detach().squeeze().cpu().numpy().astype(np.float32)
        snac_model.to("cuda")  # move back for next iteration

        # Save
        sf.write(wav_path, audio_np, TARGET_SAMPLE_RATE)
        gen_time = time.time() - start_time
        audio_duration = len(audio_np) / TARGET_SAMPLE_RATE

        print(f"    ✅ Saved: {wav_filename} ({audio_duration:.1f}s audio, generated in {gen_time:.1f}s)")
        display(Audio(audio_np, rate=TARGET_SAMPLE_RATE))

    except Exception as e:
        gen_time = time.time() - start_time
        status = "error"
        error_msg = str(e)
        audio_duration = 0
        print(f"    ❌ Error: {e}")
        # Try to recover GPU memory
        torch.cuda.empty_cache()
        gc.collect()
        try:
            snac_model.to("cuda")
        except:
            pass

    results.append({
        "id": test_id,
        "text": text,
        "language": test["language"],
        "category": test["category"],
        "wav_file": wav_filename,
        "generation_time_sec": round(gen_time, 2),
        "audio_duration_sec": round(audio_duration, 2),
        "status": status,
        "error": error_msg
    })

# Save metadata
metadata = {"model": "orpheus", "model_name": MODEL_NAME, "results": results}
meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n{'='*70}")
print(f"  BATCH COMPLETE: {sum(1 for r in results if r['status']=='success')}/{len(results)} successful")
print(f"  Metadata saved: {meta_path}")
print(f"{'='*70}")

## 9. Download Results

In [ ]:
import shutil

zip_name = "orpheus_outputs"
shutil.make_archive(zip_name, "zip", "outputs/orpheus")
print(f"✅ Created {zip_name}.zip")

# Summary table
print(f"\n{'ID':>3} | {'Status':>7} | {'Time':>6} | {'Duration':>8} | {'Language':>8} | {'Category':<22} | Text")
print("-" * 110)
for r in results:
    print(f"{r['id']:3d} | {r['status']:>7} | {r['generation_time_sec']:5.1f}s | {r['audio_duration_sec']:6.1f}s | {r['language']:>8} | {r['category']:<22} | {r['text'][:40]}...")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
except:
    print(f"\nDownload {zip_name}.zip from the file browser.")